# Shadow-feature scenario

Parameterized notebook. One run = one scenario. Writes `results/<scenario_id>.json` and shows the per-feature importance distribution for this scenario.

In [ ]:
# Default parameters; papermill overrides via the `parameters` tag.
scenario_id = 'smoke-default'
regime = 'A'
n = 1000
p = 0.10
delta = 0.5
n_real = 5
n_noise = 5
control = False
backend = 'xgboost'
n_repeats = 5
results_dir = 'results'


In [ ]:
import json, os, sys, time, platform
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, xgboost

sys.path.insert(0, os.getcwd())
from shadow_lib import derive_seed, make_dataset, compute_metrics, aggregate_repeats

effective_delta = 0.0 if control else float(delta)
n_min = int(round(n * p))
print(f'scenario_id={scenario_id} regime={regime} n={n} p={p} delta={delta} '
      f'control={control} backend={backend} n_repeats={n_repeats} n_min={n_min}')


In [ ]:
def make_clf(backend, seed):
    if backend == 'xgboost':
        from xgboost import XGBClassifier
        return XGBClassifier(
            n_estimators=300, max_depth=6, learning_rate=0.1,
            tree_method='hist', importance_type='gain', n_jobs=-1,
            eval_metric='logloss', verbosity=0, random_state=seed,
        )
    if backend == 'sklearn_gbm':
        from sklearn.ensemble import GradientBoostingClassifier
        return GradientBoostingClassifier(
            n_estimators=200, max_depth=3, learning_rate=0.1,
            random_state=seed,
        )
    raise ValueError(f'unknown backend: {backend!r}')


In [ ]:
per_repeat = {k: [] for k in
    ['naive_floor', 'p90_floor', 'mean_real_importance', 'fnr_max', 'fnr_p90']}
importances_all = []
wall_start = time.time()
for r in range(n_repeats):
    seed = derive_seed(scenario_id, r)
    X, y = make_dataset(n=n, p=p, delta=effective_delta,
                       n_real=n_real, n_noise=n_noise, seed=seed)
    clf = make_clf(backend, seed)
    clf.fit(X, y)
    imp = np.asarray(clf.feature_importances_, dtype=float)
    importances_all.append(imp)
    m = compute_metrics(imp, n_real=n_real, n_noise=n_noise)
    for k, v in m.items():
        per_repeat[k].append(v)
wall_time = time.time() - wall_start
print(f'wall_time={wall_time:.2f}s')


In [ ]:
aggregate = aggregate_repeats(per_repeat)
lib_versions = {
    'python': platform.python_version(),
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'sklearn': sklearn.__version__,
    'xgboost': xgboost.__version__,
}
result = {
    'scenario_id': scenario_id,
    'params': {'regime': regime, 'n': n, 'p': p, 'delta': delta,
               'n_real': n_real, 'n_noise': n_noise, 'control': control,
               'backend': backend, 'n_repeats': n_repeats},
    'effective_delta': effective_delta,
    'n_min': n_min,
    'n_repeats_used': n_repeats,
    'wall_time_seconds': wall_time,
    'library_versions': lib_versions,
    'per_repeat': per_repeat,
    'aggregate': aggregate,
}
Path(results_dir).mkdir(parents=True, exist_ok=True)
out_path = Path(results_dir) / f'{scenario_id}.json'
with open(out_path, 'w') as f:
    json.dump(result, f, indent=2)
print('wrote', out_path)


In [ ]:
# Inline plot: per-feature importance distribution across repeats.
imp_matrix = np.array(importances_all)
row_sums = imp_matrix.sum(axis=1, keepdims=True)
row_sums = np.where(row_sums > 0, row_sums, 1.0)
imp_norm = imp_matrix / row_sums
labels = [f'real_{i}' for i in range(n_real)] + [f'noise_{j}' for j in range(n_noise)]
fig, ax = plt.subplots(figsize=(8, 4))
positions = np.arange(n_real + n_noise)
ax.boxplot(imp_norm, positions=positions, widths=0.6, showfliers=False)
ax.axhline(aggregate['naive_floor']['mean'], color='red', linestyle='--',
           label=f"naive floor (mean={aggregate['naive_floor']['mean']:.3f})")
ax.axhline(aggregate['p90_floor']['mean'], color='orange', linestyle=':',
           label=f"p90 floor (mean={aggregate['p90_floor']['mean']:.3f})")
ax.set_xticks(positions)
ax.set_xticklabels(labels, rotation=45)
ax.set_ylabel('normalized gain importance')
ax.set_title(f'{scenario_id} | n={n} p={p} delta={delta} control={control} n_min={n_min}')
ax.legend()
fig.tight_layout()
plt.show()


In [ ]:
print('FNR_max (mean):', aggregate['fnr_max']['mean'])
print('FNR_p90 (mean):', aggregate['fnr_p90']['mean'])
print('Noise floor (mean):', aggregate['naive_floor']['mean'])
print('Mean real importance (mean):', aggregate['mean_real_importance']['mean'])
